# FarmLens — Step 2: Fine-tune Airavata on KCC (Unsloth + QLoRA)

Fine-tunes the Hindi-first **Airavata** (Llama-2-7B) base on the **already-translated** KCC Hindi Q&A from Step 1 — giving natural Hindi farming *tone*.

> Facts still come from **RAG** in the FarmLens app — this fine-tune only shapes *language/tone*, never facts.

## Before you run
1. **Settings → Accelerator → GPU** (T4 x2 or P100). Airavata-7B in 4-bit fits a single T4.
2. **Add Input** → the `kcc-hindi-translated` dataset you saved in Step 1.
3. Set `TRANSLATED_CSV` in Config to its path under `/kaggle/input/...`.

**Outputs:** a small LoRA adapter + a quantized **GGUF** you import into Ollama.

In [ ]:
%%capture
# Official Kaggle install — pinned deps avoid pip's slow resolver backtracking.
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q hf_transfer

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"   # faster model download

In [ ]:
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported

# ───────────────────────── Config ─────────────────────────
MODEL_NAME     = "ai4bharat/Airavata"   # Hindi-first Llama-2-7B base (MVP)
# Phase-2 multilingual alternatives (just swap MODEL_NAME):
#   "unsloth/llama-3.1-8b-Instruct-bnb-4bit"   # newer, multilingual, Unsloth-optimized
#   "sarvamai/sarvam-1"                          # 10 Indic languages, 2B, lightweight
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT   = True

# Translated KCC from Step 1 — point this at the dataset you added
TRANSLATED_CSV = "/kaggle/input/kcc-hindi-translated/kcc_translated.csv"  # <-- CHANGE

# Training / output
NUM_EPOCHS    = 1
LEARNING_RATE = 2e-4
OUTPUT_LORA   = "farmlens-airavata-lora"
OUTPUT_GGUF   = "farmlens-airavata-gguf"

In [ ]:
# Load the base model in 4-bit and attach LoRA adapters
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("hf-token")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,          # auto: bf16 if supported else fp16
    load_in_4bit   = LOAD_IN_4BIT,
    token= secret_value_0,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,                         # 0 is optimized in Unsloth
    bias = "none",
    use_gradient_checkpointing = "unsloth",   # long-context friendly, less VRAM
    random_state = 3407,
)
model.print_trainable_parameters()

## 1. Load the translated (Hindi) KCC data

Reads the CSV from Step 1 and formats each row into the instruction prompt.

In [ ]:
import pandas as pd
from datasets import Dataset

PROMPT = """### सवाल:
{question}

### उत्तर:
{answer}"""

EOS = tokenizer.eos_token

df = pd.read_csv(TRANSLATED_CSV).dropna()
df["text"] = df.apply(
    lambda r: PROMPT.format(question=r["question"], answer=r["answer"]) + EOS, axis=1
)
dataset = Dataset.from_pandas(df[["text"]], preserve_index=False)
print("Training examples:", len(dataset))
print("\n--- sample ---\n", dataset[0]["text"][:400])

## 2. Train (SFT)

> If TRL raises a signature error on `SFTConfig` (its API changes often), copy the trainer cell from the latest official Unsloth Kaggle notebook — the rest is unaffected.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = 512,    # KCC Q&A are short — 512 ≫ 2048 for speed
        packing                     = True,   # pack short examples → far less padding waste
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,      # effective batch = 16
        warmup_steps                = 5,
        max_steps                   = 300,    # fast run (~15-25 min). Set to -1 to use epochs.
        num_train_epochs            = NUM_EPOCHS,  # only used when max_steps = -1
        learning_rate               = LEARNING_RATE,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps     = 20,
        optim             = "adamw_8bit",
        weight_decay      = 0.01,
        lr_scheduler_type = "linear",
        seed              = 3407,
        output_dir        = "outputs",
        report_to         = "none",
    ),
)
trainer_stats = trainer.train()
print("Training done:", trainer_stats.metrics)

## 3. Quick inference test

In [ ]:
FastLanguageModel.for_inference(model)   # 2x faster generation


def ask(question: str, max_new_tokens: int = 220) -> str:
    prompt = PROMPT.format(question=question, answer="")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=0.7, top_p=0.9, do_sample=True,
    )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split("### उत्तर:")[-1].strip()


print(ask("गेहूं की फसल में पीला रतुआ रोग कैसे रोकें?"))
print("\n---\n")
print(ask("धान की बुवाई का सही समय क्या है?"))

## 4. Save the LoRA adapter + push to HF (safeguard)

> 💾 This saves the adapter locally **and immediately pushes it to Hugging Face** — because a Kaggle session reset wipes `/kaggle/working`. The LoRA is your trained artifact; with it on HF you can always re-export the GGUF later **without retraining**. Set `HF_TOKEN` (a **write** token) as a Kaggle Secret first.

In [ ]:
model.save_pretrained(OUTPUT_LORA)
tokenizer.save_pretrained(OUTPUT_LORA)
print("Saved LoRA adapter to", OUTPUT_LORA)

# Push to HF IMMEDIATELY — a Kaggle session reset wipes /kaggle/working, so don't
# wait until the end. The adapter is tiny (~100 MB), so this takes seconds.
LORA_REPO = "YOUR-USERNAME/farmlens-airavata-lora"   # <-- change to your HF username
model.push_to_hub(LORA_REPO, token=secret_value_0)
tokenizer.push_to_hub(LORA_REPO, token=secret_value_0)
print("Pushed LoRA to https://huggingface.co/" + LORA_REPO)

## 5. Export GGUF for Ollama

Builds a quantized `q4_k_m` GGUF (~4.4 GB). Takes several minutes — Unsloth compiles llama.cpp under the hood.

> ⚠️ **Kaggle disk:** GGUF export needs ~28 GB of intermediates, but `/kaggle/working` is capped at ~20 GB. This cell frees disk, runs the conversion under `/tmp` (which is off the quota), then copies only the final `q4_k_m` back to `/kaggle/working` for download.

In [ ]:
import os, shutil, glob

# Kaggle caps /kaggle/working at ~20 GB, but GGUF export needs the merged
# safetensors (~14 GB) AND the F16 intermediate (~13.7 GB) on disk at once
# (~28 GB). So: free disk, convert under /tmp (off the quota), copy only the
# final ~4.4 GB q4_k_m back for download.

# 1. Free /kaggle/working — big disposables only (NEVER the LoRA adapter)
for name in ["farmlens-airavata-gguf", "Airavata.F16.gguf",
             "unsloth_compiled_cache", "huggingface_tokenizers_cache", "outputs"]:
    if os.path.isdir(name):
        shutil.rmtree(name, ignore_errors=True)
    elif os.path.exists(name):
        os.remove(name)
print(os.popen("df -h /tmp /kaggle/working").read())   # sanity-check free space

# 2. Convert entirely under /tmp (chdir so the F16 intermediate lands there too)
os.makedirs("/tmp/gguf", exist_ok=True)
os.chdir("/tmp/gguf")
model.save_pretrained_gguf("/tmp/gguf/out", tokenizer, quantization_method="q4_k_m")

# 3. Copy ONLY the final q4_k_m file back to working for download
os.chdir("/kaggle/working")
for f in glob.glob("/tmp/gguf/**/*.gguf", recursive=True):
    if "q4_k_m" in f.lower():
        shutil.copy(f, "/kaggle/working/")
        print("Saved to /kaggle/working:", os.path.basename(f))

# Optional: push GGUF straight to HF (still uses /tmp for the F16 step above)
# model.push_to_hub_gguf(
#     "your-username/farmlens-airavata-gguf", tokenizer,
#     quantization_method="q4_k_m", token="hf_...",
# )

## 6. Upload the GGUF to Hugging Face (reliable download)

Kaggle's Output panel often fails to download multi-GB files. Pushing to the HF Hub is **resumable and robust** — and doubles as a cloud backup.

**Setup once:** create a free HF account → **Settings → Access Tokens → a `write` token** → add it as a Kaggle Secret named `HF_TOKEN` (**Add-ons → Secrets**). Then set `repo_id` to your username below.

Afterwards, download it on your PC (resumable):
```
huggingface-cli download YOUR-USERNAME/farmlens-airavata-gguf Airavata.Q4_K_M.gguf --local-dir .
```

In [ ]:
from huggingface_hub import HfApi
repo_id  = "YOUR-USERNAME/farmlens-airavata-gguf"        # <-- change to your HF username

api = HfApi(token=secret_value_0)
api.create_repo(repo_id, repo_type="model", exist_ok=True)
api.upload_file(
    path_or_fileobj="/kaggle/working/Airavata.Q4_K_M.gguf",
    path_in_repo="Airavata.Q4_K_M.gguf",
    repo_id=repo_id,
    repo_type="model",
)
print("Uploaded → https://huggingface.co/" + repo_id)

## 7. Import into Ollama (in FarmLens)

1. Get `Airavata.Q4_K_M.gguf` onto your PC — either via the HF download above, or from this notebook's **Output** panel.
2. Put it next to a file named `Modelfile`:

```
FROM ./Airavata.Q4_K_M.gguf

TEMPLATE """### सवाल:
{{ .Prompt }}

### उत्तर:
"""
PARAMETER stop "### सवाल:"
PARAMETER temperature 0.7
```

3. Create and test the model:

```
ollama create farmlens -f Modelfile
ollama run farmlens "गेहूं में कौन सी खाद डालें?"
```

4. Point FarmLens at it — in `.env`:

```
OLLAMA_MODEL=farmlens
```

> Reminder: RAG supplies the *facts* (ICAR docs); this model supplies the Hindi farming *tone*.